In [2]:
import cobra
import sys
# sys.path.insert(1, '../scripts/')
# from human_me.utils import *

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


In [3]:
biomass_m = [m for m in human_model.metabolites if 'biomass' in m.id]
biomass_r = [r for r in human_model.reactions if 'biomass' in r.id]

Let's use the Recon2.2 default reactions as an example. 
The biomass demand reaction is:

In [59]:
human_model.reactions.get_by_id('biomass_reaction').reaction

'0.014 biomass_DNA[c] + 0.058 biomass_RNA[c] + 0.071 biomass_carbohydrate[c] + 0.097 biomass_lipid[c] + 0.054 biomass_other[c] + 0.706 biomass_protein[c] --> '

Here, each stoichiometric coefficient results in a fixed proportion of each biomass component (gram substrate per grame dry-weight cell). In this case, the substrates are in units of grams. So, the mmol of biomass produced is equivalent to 1 g of biomass (1gDw,cell), s.t. the flux through biomass simply becomes hr^-1 (from mmol/gDw/hr = gDw/gDw/hr).
<br>
Looking into the formation of lipids, we see that this is further divided into proportions:

In [61]:
human_model.reactions.get_by_id('biomass_lipid').reaction

'0.210319587628866 chsterol[c] + 0.120185567010309 clpn_hs[c] + 0.240360824742268 pail_hs[c] + 1.59237113402062 pchol_hs[c] + 0.570865979381443 pe_hs[c] + 0.0300412371134021 pglyc_hs[c] + 0.0600927835051546 ps_hs[c] + 0.180268041237113 sphmyln_hs[c] --> biomass_lipid[c]'

For example, you have chsterol[c] contributing to 21% of lipid biomass. To maintain 1) a constant proportion of lipid biomass (0.097) and 2) a constant proportion of its constituents (e.g., 0.21 chsterol), the coefficient of the constitutents (units of mmol) must be scaled by their molecular weight (units of g/mmol or kDa) and the product must contain the final proportion. Constrainting this reaction flux by growth maintains the final proportion as a function of growth. 
<br> 


In [70]:
print('0.21*MW(chsterol)*chsterol[c] + 0.12 MW(clpn_hs)*clpnb_hs[c] + ... +  <--> 0.097 biomass_lipid[c] \t Flux bounds: [mu, mu]')
print('\n biomass_lipid --> biomass \t flux bounds: [0,1000] ')


0.21*MW(chsterol)*chsterol[c] + 0.12 MW(clpn_hs)*clpnb_hs[c] + ... +  <--> 0.097 biomass_lipid[c] 	 Flux bounds: [mu, mu]

 biomass_lipid --> biomass 	 flux bounds: [0,1000] 


For variable biomass (just protein and RNA for now), biomass is produced in each expression reaction with a coefficient equal to the molecular weight of the protein/RNA (units of kDa or g/mmol). In this way, the final reaction flux in units of mmol/gDw/hr again scales to hr^-1 

In [71]:
print('Translation: X aa --> protein_A + MW(protein_A)*biomass_protein')
print('\n biomass_protein --> biomass')

Translation: X aa --> protein_A + MW(protein_A)*biomass_protein

 biomass_protein --> biomass


And finally, there is a biomass dilution reaction which equals the growth rate 

In [72]:
print('biomass --> ')

biomass --> 


In [ ]:
# 'MW_chsterol[c]*chsterol[c]  <--> 0.21* 0.097 biomass_lipid[c]' [mu,mu]
# 'MW_clpn_hs[c]*clpnb_hs[c] <--> 0.12*0.097 biomass_lipid[c]''

In [ ]:
# '0.21*MW_chsterol[c]*chsterol[c] + 0.12 MW_clpn_hs[c]*clpnb_hs[c] + ... +  <--> 0.097 biomass_lipid[c]' [mu,mu]
# 'biomass_lipid --> biomass' [0,1000] # scaling and mu bounds keeps fixed proportions 

# 'X aa --> protein A + MW_proteinA*protein_biomass' [0,1e3]
# 'protein_biomass --> biomass' [0,1e3] # allow variability 


# 'X ntps --> mRNA A + MW_RNA*mRNA_biomass'
# 'mRNA_biomass --> RNA_biomass'
# 'RNA_biomass --> biomass'

# 'biomass --> ' # dilution 1g/gDW*hr = hr^-1
# # constrained to 0.058 + 0.706